# NBA Fantasy Points Predictor

Predicting DraftKings fantasy points per player per game using recent game history.

DK scoring: PTS + 1.25*REB + 1.5*AST + 2*STL + 2*BLK - 0.5*TOV + 0.5*FG3M

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

from pathlib import Path

DATA_DIR = Path("data")

## Data Loading

In [2]:
player_logs = pd.read_csv(DATA_DIR / "nba_player_game_logs.csv")
games       = pd.read_csv(DATA_DIR / "nba_historical_games.csv")

# Parse dates
player_logs["GAME_DATE"] = pd.to_datetime(player_logs["GAME_DATE"])
games["date"]            = pd.to_datetime(games["date"])

# Normalize GAME_ID to int for joining (player logs have leading zeros: '0029900001')
player_logs["game_id_int"] = player_logs["GAME_ID"].astype(int)
games["game_id_int"]       = games["game_id"].astype(int)

# Convert MIN to numeric (NBA API occasionally returns "30:45" string format)
player_logs["MIN"] = pd.to_numeric(player_logs["MIN"], errors="coerce")

print("Player logs:", player_logs.shape)
print("Games:      ", games.shape)
player_logs.head(3)

Player logs: (673733, 25)
Games:       (32444, 47)


,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GAME_ID,GAME_DATE,MIN,PTS,REB,...,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,PLUS_MINUS,game_id_int
0,21999,431,Shawn Kemp,1610612739,CLE,29900001,1999-11-02,31,17,5,...,20,0.300,0,0,NaN,5,8,0.625,2,29900001
1,21999,95,Mark Bryant,1610612739,CLE,29900001,1999-11-02,15,3,3,...,3,0.333,0,0,NaN,1,2,0.500,-24,29900001
2,21999,1538,Cedric Henderson,1610612739,CLE,29900001,1999-11-02,20,0,2,...,1,0.000,0,0,NaN,0,0,NaN,-17,29900001


## Target

DK scoring: PTS x 1.0, REB x 1.25, AST x 1.5, STL x 2.0, BLK x 2.0, TOV x -0.5, FG3M x 0.5

In [3]:
player_logs["FANTASY_PTS"] = (
    player_logs["PTS"]  * 1.00 +
    player_logs["REB"]  * 1.25 +
    player_logs["AST"]  * 1.50 +
    player_logs["STL"]  * 2.00 +
    player_logs["BLK"]  * 2.00 +
    player_logs["TOV"]  * -0.50 +
    player_logs["FG3M"] * 0.50
)

print(f"Fantasy points - mean: {player_logs['FANTASY_PTS'].mean():.2f}, "
      f"std: {player_logs['FANTASY_PTS'].std():.2f}, "
      f"min: {player_logs['FANTASY_PTS'].min():.2f}, "
      f"max: {player_logs['FANTASY_PTS'].max():.2f}")

Fantasy points - mean: 20.55, std: 14.15, min: -2.00, max: 107.75


## Feature Engineering

Using the last 10 games as features for each player, their team, and the opponent.

### Player stats (lags 1-10)

In [4]:
PLAYER_STATS = ["PTS", "REB", "AST", "STL", "BLK", "TOV", "FG3M", "MIN"]

# sort first or shift() won't look back in time correctly
player_logs = player_logs.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)

player_lag_cols = []
for stat in PLAYER_STATS:
    for lag in range(1, 11):
        col = f"player_{stat}_lag{lag}"
        player_logs[col] = player_logs.groupby("PLAYER_ID")[stat].shift(lag)
        player_lag_cols.append(col)

print(f"player lag features: {len(player_lag_cols)}")
player_logs[["PLAYER_NAME", "GAME_DATE"] + player_lag_cols[:4]].head()

player lag features: 80


,PLAYER_NAME,GAME_DATE,player_PTS_lag1,player_PTS_lag2,player_PTS_lag3,player_PTS_lag4
0,Grant Long,1999-12-27,NaN,NaN,NaN,NaN
1,Grant Long,1999-12-29,4.0,NaN,NaN,NaN
2,Grant Long,1999-12-30,4.0,4.0,NaN,NaN
3,Grant Long,2000-01-04,4.0,4.0,4.0,NaN
4,Grant Long,2000-01-05,1.0,4.0,4.0,4.0


### Team stats (lags 1-10)

The games table has separate home/away columns so I stack them into one row per team first.

In [5]:
TEAM_STATS = ["score", "fg_made", "fg3_made", "reb", "ast", "stl", "blk", "tov"]

home_games = games[["game_id_int", "date", "home_team",
                     "home_score", "home_fg_made", "home_fg3_made",
                     "home_reb", "home_ast", "home_stl",
                     "home_blk", "home_tov"]].copy()
home_games.columns = ["game_id_int", "date", "team"] + TEAM_STATS

away_games = games[["game_id_int", "date", "away_team",
                     "away_score", "away_fg_made", "away_fg3_made",
                     "away_reb", "away_ast", "away_stl",
                     "away_blk", "away_tov"]].copy()
away_games.columns = ["game_id_int", "date", "team"] + TEAM_STATS

team_games = pd.concat([home_games, away_games], ignore_index=True)
team_games = team_games.sort_values(["team", "date"]).reset_index(drop=True)

print(f"Team-game rows (should be 2 x {len(games):,}): {len(team_games):,}")
team_games.head(3)

Team-game rows (should be 2 x 32,444): 64,888


,game_id_int,date,team,score,fg_made,fg3_made,reb,ast,stl,blk,tov
0,29900003,1999-11-02,ATL,87,31,2,50,15,5,5,23
1,29900021,1999-11-04,ATL,109,41,6,46,22,3,5,26
2,29900037,1999-11-06,ATL,113,44,3,42,21,10,3,12


In [6]:
team_lag_cols = []
for stat in TEAM_STATS:
    for lag in range(1, 11):
        col = f"team_{stat}_lag{lag}"
        team_games[col] = team_games.groupby("team")[stat].shift(lag)
        team_lag_cols.append(col)

print(f"team lag features: {len(team_lag_cols)}")

team lag features: 80


### Opponent stats (lags 1-10)

In [7]:
# figure out who each team played in each game
opp_map = pd.concat([
    games[["game_id_int", "home_team", "away_team"]].rename(
        columns={"home_team": "team", "away_team": "opponent"}),
    games[["game_id_int", "away_team", "home_team"]].rename(
        columns={"away_team": "team", "home_team": "opponent"})
], ignore_index=True)

team_games = team_games.merge(opp_map, on=["game_id_int", "team"], how="left")
team_games[["team", "opponent", "date"]].head(3)

,team,opponent,date
0,ATL,WAS,1999-11-02
1,ATL,MIL,1999-11-04
2,ATL,CHI,1999-11-06


In [8]:
# look up each opponent's lag features for the same game
opp_lag_cols = [f"opp_{s}_lag{l}" for s in TEAM_STATS for l in range(1, 11)]

opp_lookup = team_games[["game_id_int", "team"] + team_lag_cols].rename(
    columns={"team": "opponent"}
)
opp_lookup.columns = ["game_id_int", "opponent"] + opp_lag_cols

team_games = team_games.merge(opp_lookup, on=["game_id_int", "opponent"], how="left")
print(f"opp lag features: {len(opp_lag_cols)}")

opp lag features: 80


### Build feature matrix

Merge player + team + opponent lags. Drop rows missing lag history and short appearances (< 10 min).

In [9]:
team_cols = ["game_id_int", "team"] + team_lag_cols + opp_lag_cols

df = player_logs.merge(
    team_games[team_cols],
    left_on=["game_id_int", "TEAM_ABBREVIATION"],
    right_on=["game_id_int", "team"],
    how="inner"
)

all_feature_cols = player_lag_cols + team_lag_cols + opp_lag_cols

df_clean = df.dropna(subset=all_feature_cols).reset_index(drop=True)
df_clean = df_clean[df_clean["MIN"] >= 10].reset_index(drop=True)

print(f"{len(df_clean):,} rows, {len(all_feature_cols)} features")

X = df_clean[all_feature_cols]
y = df_clean["FANTASY_PTS"]

562,632 rows, 240 features


## Train/Test Split

Splitting by date so we're not training on future games. Everything before 2023-24 is train, rest is test.

In [10]:
train_mask = df_clean["GAME_DATE"] < "2023-10-01"

X_train = X[train_mask]
X_test  = X[~train_mask]
y_train = y[train_mask]
y_test  = y[~train_mask]

print(f"Train: {len(X_train):,} rows, Test: {len(X_test):,} rows")

Train: 496,750 rows, Test: 65,882 rows


## Linear Regression Baseline

Using TimeSeriesSplit for CV so future folds don't leak into training.

In [11]:
# 1. choose model class (LinearRegression, already imported above)
# 2. instantiate model
model = make_pipeline(StandardScaler(), LinearRegression())

# 3. fit model to data
model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearregression', LinearRegression())])

In [12]:
# 4. predict on training/testing data, evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse  = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))

print(f"Train RMSE: {train_rmse:.3f} fantasy points")
print(f"Test  RMSE: {test_rmse:.3f} fantasy points")

Train RMSE: 9.427 fantasy points
Test  RMSE: 9.895 fantasy points


In [13]:
tscv = TimeSeriesSplit(n_splits=5)

cv_rmses = np.sqrt(
    -cross_val_score(model, X_train, y_train, cv=tscv,
                     scoring="neg_mean_squared_error")
)

print(f"CV RMSE (TimeSeriesSplit): {cv_rmses.mean():.3f} +/- {cv_rmses.std():.3f}")
print(f"per fold: {np.round(cv_rmses, 3)}")

CV RMSE (TimeSeriesSplit): 9.460 +/- 0.069
per fold: [9.48  9.343 9.455 9.465 9.557]


### Additional Features

Rolling averages, hot/cold trends, game context, and efficiency stats.

In [14]:
# rolling window averages built from the existing lag columns
WINDOWS = [3, 5, 10]

roll_cols = []
for stat in PLAYER_STATS:
    for w in WINDOWS:
        col = f"player_{stat}_L{w}"
        df_clean[col] = df_clean[[f"player_{stat}_lag{l}" for l in range(1, w+1)]].mean(axis=1)
        roll_cols.append(col)

for stat in TEAM_STATS:
    for w in WINDOWS:
        col = f"team_{stat}_L{w}"
        df_clean[col] = df_clean[[f"team_{stat}_lag{l}" for l in range(1, w+1)]].mean(axis=1)
        roll_cols.append(col)

for stat in TEAM_STATS:
    for w in WINDOWS:
        col = f"opp_{stat}_L{w}"
        df_clean[col] = df_clean[[f"opp_{stat}_lag{l}" for l in range(1, w+1)]].mean(axis=1)
        roll_cols.append(col)

print(f"rolling avg features: {len(roll_cols)}")

rolling avg features: 72


In [15]:
# hot/cold trend: L3 avg minus L10 avg for each player stat
trend_cols = []
for stat in PLAYER_STATS:
    col = f"player_{stat}_trend"
    df_clean[col] = df_clean[f"player_{stat}_L3"] - df_clean[f"player_{stat}_L10"]
    trend_cols.append(col)

print(f"trend features: {len(trend_cols)}")

trend features: 8


In [16]:
# is_home
home_lookup = games[["game_id_int", "home_team"]].copy()
home_lookup.columns = ["game_id_int", "_home"]
df_clean = df_clean.merge(home_lookup, on="game_id_int", how="left")
df_clean["is_home"] = (df_clean["TEAM_ABBREVIATION"] == df_clean["_home"]).astype(int)
df_clean.drop(columns=["_home"], inplace=True)

# days rest per team - one row per unique (team, game) then merge back
team_sched = (df_clean[["TEAM_ABBREVIATION", "GAME_DATE", "game_id_int"]]
              .drop_duplicates(subset=["TEAM_ABBREVIATION", "game_id_int"])
              .sort_values(["TEAM_ABBREVIATION", "GAME_DATE"])
              .reset_index(drop=True))
team_sched["days_rest"] = (team_sched.groupby("TEAM_ABBREVIATION")["GAME_DATE"]
                           .diff().dt.days.clip(upper=7).fillna(3).astype(int))
df_clean = df_clean.merge(
    team_sched[["game_id_int", "TEAM_ABBREVIATION", "days_rest"]],
    on=["game_id_int", "TEAM_ABBREVIATION"], how="left"
)

# opponent rest days
opp_key = pd.concat([
    games[["game_id_int", "home_team", "away_team"]].rename(
        columns={"home_team": "TEAM_ABBREVIATION", "away_team": "_opp"}),
    games[["game_id_int", "away_team", "home_team"]].rename(
        columns={"away_team": "TEAM_ABBREVIATION", "home_team": "_opp"})
], ignore_index=True)
opp_rest = team_sched[["game_id_int", "TEAM_ABBREVIATION", "days_rest"]].rename(
    columns={"TEAM_ABBREVIATION": "_opp", "days_rest": "opp_days_rest"}
)
opp_key = opp_key.merge(opp_rest, on=["game_id_int", "_opp"], how="left")
df_clean = df_clean.merge(
    opp_key[["game_id_int", "TEAM_ABBREVIATION", "opp_days_rest"]],
    on=["game_id_int", "TEAM_ABBREVIATION"], how="left"
)

context_cols = ["is_home", "days_rest", "opp_days_rest"]
print("context features:", context_cols)

context features: ['is_home', 'days_rest', 'opp_days_rest']


In [17]:
EFF_STATS = ["FGA", "FTA", "FG_PCT", "FG3_PCT", "PLUS_MINUS"]

# fill NaN shooting percentages before rolling (0 attempts = 0%)
player_logs["FG_PCT"]  = player_logs["FG_PCT"].fillna(0)
player_logs["FG3_PCT"] = player_logs["FG3_PCT"].fillna(0)

eff_cols = []
for stat in EFF_STATS:
    for w in WINDOWS:
        col = f"player_{stat}_L{w}"
        player_logs[col] = (player_logs.groupby("PLAYER_ID")[stat]
                           .transform(lambda x: x.shift(1).rolling(w, min_periods=w).mean()))
        eff_cols.append(col)

df_clean = df_clean.merge(
    player_logs[["PLAYER_ID", "game_id_int"] + eff_cols],
    on=["PLAYER_ID", "game_id_int"], how="left"
)
print(f"efficiency features: {len(eff_cols)}")

C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\1275467999.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs[col] = (player_logs.groupby("PLAYER_ID")[stat]


C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\1275467999.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs[col] = (player_logs.groupby("PLAYER_ID")[stat]


efficiency features: 15


## External Data & Advanced Features

Going beyond box-score history. The next sections add:
- **Missing teammates** - rotation minutes absent from the box score (proxy for injury/rest)
- **Schedule density** - back-to-backs and games in the last 7 days
- **Player position** - from `nba_api` CommonPlayerInfo
- **Pace + advanced ratings** - from BoxScoreAdvancedV2 (team & opponent)
- **Defense vs Position (DvP)** - rolling FP allowed by opponent to player's position

Cells skip gracefully if a data file isn't present (collection scripts may still be running).

In [18]:
# Missing-teammates feature
# When key rotation players are out, remaining players see usage spikes.
# We can detect this without an injury feed: each (team, game), look at the
# team's typical top minute-getters from L10 and check who's missing from this box.

# Step 1: each player-game -> rolling L10 avg MIN (excluding current game)
player_logs["MIN_L10"] = (player_logs.groupby("PLAYER_ID")["MIN"]
                          .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean()))

# Step 2: for each (team, game), sum the L10 MIN of players in the box score
team_game_min = (player_logs.dropna(subset=["MIN_L10"])
                 .groupby(["game_id_int", "TEAM_ABBREVIATION"])["MIN_L10"]
                 .agg(["sum", "count"])
                 .reset_index()
                 .rename(columns={"sum": "team_l10_min_played", "count": "team_players_played"}))

# Step 3: rolling avg of team_l10_min_played across the team's last 10 games
# = baseline expected, so deficit = baseline - actual
team_game_min = team_game_min.merge(
    games[["game_id_int", "date"]], on="game_id_int", how="left"
).sort_values(["TEAM_ABBREVIATION", "date"]).reset_index(drop=True)

team_game_min["team_l10_min_baseline"] = (
    team_game_min.groupby("TEAM_ABBREVIATION")["team_l10_min_played"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)
team_game_min["missing_min_deficit"] = (
    team_game_min["team_l10_min_baseline"] - team_game_min["team_l10_min_played"]
)

# Step 4: merge into df_clean
df_clean = df_clean.merge(
    team_game_min[["game_id_int", "TEAM_ABBREVIATION",
                   "team_l10_min_played", "team_players_played", "missing_min_deficit"]],
    on=["game_id_int", "TEAM_ABBREVIATION"], how="left"
)

missing_cols = ["team_l10_min_played", "team_players_played", "missing_min_deficit"]
print(f"missing-teammates features: {missing_cols}")
print(df_clean[missing_cols].describe().to_string())

C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\2489024629.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs["MIN_L10"] = (player_logs.groupby("PLAYER_ID")["MIN"]


missing-teammates features: ['team_l10_min_played', 'team_players_played', 'missing_min_deficit']
       team_l10_min_played  team_players_played  missing_min_deficit
count        562632.000000        562632.000000        562632.000000
mean            241.157980            10.351813            -0.626383
std              24.554682             1.307570            25.807486
min              55.100000             4.000000          -169.008333
25%             227.100000             9.000000           -16.214841
50%             242.000000            10.000000            -0.708095
75%             256.300000            11.000000            14.660000
max             380.400000            15.000000           189.565238


In [19]:
# Schedule-density features: back-to-back, games in last 4 / 7 days
# team_sched was built earlier in the context-feats cell
team_sched = team_sched.sort_values(["TEAM_ABBREVIATION", "GAME_DATE"]).reset_index(drop=True)

# back-to-back: previous game was 1 day before
team_sched["is_b2b"] = (team_sched["days_rest"] == 1).astype(int)

# count games in last N days (excluding today)
def games_in_last_n_days(group, n):
    dates = group["GAME_DATE"].values.astype("datetime64[D]")
    out = np.zeros(len(dates), dtype=int)
    for i in range(len(dates)):
        out[i] = ((dates < dates[i]) & (dates >= dates[i] - np.timedelta64(n, "D"))).sum()
    return pd.Series(out, index=group.index)

team_sched["games_last_4d"] = (team_sched.groupby("TEAM_ABBREVIATION", group_keys=False)
                                .apply(lambda g: games_in_last_n_days(g, 4)))
team_sched["games_last_7d"] = (team_sched.groupby("TEAM_ABBREVIATION", group_keys=False)
                                .apply(lambda g: games_in_last_n_days(g, 7)))

df_clean = df_clean.merge(
    team_sched[["game_id_int", "TEAM_ABBREVIATION", "is_b2b", "games_last_4d", "games_last_7d"]],
    on=["game_id_int", "TEAM_ABBREVIATION"], how="left"
)

schedule_cols = ["is_b2b", "games_last_4d", "games_last_7d"]
print(f"schedule density features: {schedule_cols}")
print(df_clean[schedule_cols].describe().to_string())

C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\1843870978.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: games_in_last_n_days(g, 4)))


C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\1843870978.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: games_in_last_n_days(g, 7)))


schedule density features: ['is_b2b', 'games_last_4d', 'games_last_7d']
              is_b2b  games_last_4d  games_last_7d
count  562632.000000  562632.000000  562632.000000
mean        0.217755       1.696670       3.098580
std         0.412720       0.569902       0.865733
min         0.000000       0.000000       0.000000
25%         0.000000       1.000000       3.000000
50%         0.000000       2.000000       3.000000
75%         0.000000       2.000000       4.000000
max         1.000000       3.000000       5.000000


In [20]:
# Player position from CommonPlayerInfo
# Maps NBA's full-word positions to G/F/C buckets
position_path = DATA_DIR / "nba_player_info.csv"
position_cols = []
if position_path.exists():
    pinfo = pd.read_csv(position_path)
    print(f"Loaded {len(pinfo):,} player info rows")

    def bucket_position(p):
        if not isinstance(p, str):
            return "U"
        p = p.lower()
        if "guard" in p:   return "G"
        if "center" in p:  return "C"
        if "forward" in p: return "F"
        return "U"

    pinfo["pos_bucket"] = pinfo["POSITION"].apply(bucket_position)
    pinfo = pinfo[["PERSON_ID", "pos_bucket", "HEIGHT", "DRAFT_YEAR"]].rename(
        columns={"PERSON_ID": "PLAYER_ID"})

    # height -> inches
    def height_to_inches(h):
        if not isinstance(h, str) or "-" not in h:
            return np.nan
        ft, inch = h.split("-")
        return int(ft) * 12 + int(inch)
    pinfo["height_in"] = pinfo["HEIGHT"].apply(height_to_inches)
    pinfo["draft_year"] = pd.to_numeric(pinfo["DRAFT_YEAR"], errors="coerce")
    pinfo = pinfo[["PLAYER_ID", "pos_bucket", "height_in", "draft_year"]]

    df_clean = df_clean.merge(pinfo, on="PLAYER_ID", how="left")

    # one-hot for position
    for pos in ["G", "F", "C"]:
        df_clean[f"pos_{pos}"] = (df_clean["pos_bucket"] == pos).astype(int)

    # years of experience at game time
    df_clean["years_experience"] = df_clean["GAME_DATE"].dt.year - df_clean["draft_year"]

    position_cols = ["pos_G", "pos_F", "pos_C", "height_in", "years_experience"]
    print(f"position features: {position_cols}")
    print(f"coverage: {df_clean['pos_bucket'].notna().mean()*100:.1f}% of rows have position")
else:
    print(f"No player info at {position_path} - skipping. Run collect_player_info.py first.")

Loaded 2,652 player info rows


position features: ['pos_G', 'pos_F', 'pos_C', 'height_in', 'years_experience']
coverage: 100.0% of rows have position


In [21]:
# Advanced team box score: pace, off/def rating, true shooting, etc.
# Rolling avgs over the team's last 10 games and the opponent's last 10 games.
adv_team_path = DATA_DIR / "nba_advanced_team_stats.csv"
adv_team_cols = []
if adv_team_path.exists():
    adv = pd.read_csv(adv_team_path, dtype={"GAME_ID": str})
    adv["GAME_ID"] = adv["GAME_ID"].str.zfill(10)
    adv["game_id_int"] = adv["GAME_ID"].astype(int)
    adv = adv.merge(games[["game_id_int", "date"]], on="game_id_int", how="left")
    adv = adv.sort_values(["TEAM_ABBREVIATION", "date"]).reset_index(drop=True)
    print(f"Loaded {len(adv):,} advanced team-game rows  ({adv['date'].min().date()} - {adv['date'].max().date()})")

    ADV_STATS = ["PACE", "OFF_RATING", "DEF_RATING", "TS_PCT", "EFG_PCT"]
    for stat in ADV_STATS:
        adv[f"adv_{stat}_L10"] = (adv.groupby("TEAM_ABBREVIATION")[stat]
                                  .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean()))

    # team's own rolling advanced stats
    team_keys = ["game_id_int", "TEAM_ABBREVIATION"] + [f"adv_{s}_L10" for s in ADV_STATS]
    df_clean = df_clean.merge(adv[team_keys], on=["game_id_int", "TEAM_ABBREVIATION"], how="left")

    # opponent's rolling advanced stats: rename and merge again
    opp_adv = adv[team_keys].rename(columns={
        "TEAM_ABBREVIATION": "_opp_team",
        **{f"adv_{s}_L10": f"opp_adv_{s}_L10" for s in ADV_STATS}
    })

    # need opponent map already used earlier - build a simple lookup from games
    g_opp = pd.concat([
        games[["game_id_int", "home_team", "away_team"]].rename(columns={"home_team": "TEAM_ABBREVIATION", "away_team": "_opp_team"}),
        games[["game_id_int", "away_team", "home_team"]].rename(columns={"away_team": "TEAM_ABBREVIATION", "home_team": "_opp_team"})
    ], ignore_index=True)
    opp_full = g_opp.merge(opp_adv, on=["game_id_int", "_opp_team"], how="left")
    df_clean = df_clean.merge(
        opp_full[["game_id_int", "TEAM_ABBREVIATION"] + [f"opp_adv_{s}_L10" for s in ADV_STATS]],
        on=["game_id_int", "TEAM_ABBREVIATION"], how="left"
    )

    adv_team_cols = ([f"adv_{s}_L10" for s in ADV_STATS] +
                     [f"opp_adv_{s}_L10" for s in ADV_STATS])
    print(f"advanced team features: {len(adv_team_cols)}")
    print(f"coverage: {df_clean['adv_PACE_L10'].notna().mean()*100:.1f}%")
else:
    print(f"No advanced team data at {adv_team_path} - skipping.")

No advanced team data at data\nba_advanced_team_stats.csv - skipping.


In [22]:
# Defense vs Position (DvP)
# For each (opponent, position), compute rolling FP allowed over last 20 games of that opponent vs that position.
# This is the "how leaky is this defense against PGs/SFs/Cs" signal that DFS optimizers use heavily.
dvp_cols = []
if "pos_bucket" in df_clean.columns:
    # Build opp_team for each row
    g_opp_simple = pd.concat([
        games[["game_id_int", "home_team", "away_team"]].rename(columns={"home_team": "TEAM_ABBREVIATION", "away_team": "opp_team"}),
        games[["game_id_int", "away_team", "home_team"]].rename(columns={"away_team": "TEAM_ABBREVIATION", "home_team": "opp_team"})
    ], ignore_index=True).drop_duplicates(subset=["game_id_int", "TEAM_ABBREVIATION"])

    df_clean = df_clean.merge(g_opp_simple, on=["game_id_int", "TEAM_ABBREVIATION"], how="left")

    # Group: per opponent, per opposing position, compute rolling sum of FP given up.
    # For each (game_id, opp_team, pos_bucket): sum of FP scored by players of that pos against that opp.
    dvp_base = (df_clean.dropna(subset=["pos_bucket"])
                .groupby(["game_id_int", "opp_team", "pos_bucket"])
                .agg(fp_sum=("FANTASY_PTS", "sum"), n=("FANTASY_PTS", "count"))
                .reset_index())
    dvp_base["fp_per_player"] = dvp_base["fp_sum"] / dvp_base["n"]

    dvp_base = dvp_base.merge(games[["game_id_int", "date"]], on="game_id_int", how="left")
    dvp_base = dvp_base.sort_values(["opp_team", "pos_bucket", "date"]).reset_index(drop=True)

    # Rolling avg per (opp_team, position) of FP per player
    dvp_base["dvp_L20"] = (dvp_base.groupby(["opp_team", "pos_bucket"])["fp_per_player"]
                           .transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean()))

    df_clean = df_clean.merge(
        dvp_base[["game_id_int", "opp_team", "pos_bucket", "dvp_L20"]],
        on=["game_id_int", "opp_team", "pos_bucket"], how="left"
    )

    dvp_cols = ["dvp_L20"]
    print(f"defense-vs-position features: {dvp_cols}")
    print(f"coverage: {df_clean['dvp_L20'].notna().mean()*100:.1f}%")
    print(df_clean[dvp_cols].describe().to_string())
else:
    print("No position data - skipping DvP. Run collect_player_info.py first.")

defense-vs-position features: ['dvp_L20']
coverage: 99.7%
             dvp_L20
count  561035.000000
mean       24.062284
std         2.491143
min         5.350000
25%        22.439405
50%        23.934792
75%        25.531250
max        38.375000


## Random Forest

Switching to a tree-based model. Random Forest handles non-linear interactions and correlated features naturally â€” both problems with linear regression. No StandardScaler needed; trees are scale-invariant.

In [23]:
# rolling avg of FANTASY_PTS itself - most direct signal for trees
fp_roll_cols = []
for w in WINDOWS:
    col = f"player_FANTASY_PTS_L{w}"
    player_logs[col] = (player_logs.groupby("PLAYER_ID")["FANTASY_PTS"]
                       .transform(lambda x: x.shift(1).rolling(w, min_periods=w).mean()))
    fp_roll_cols.append(col)

df_clean = df_clean.merge(
    player_logs[["PLAYER_ID", "game_id_int"] + fp_roll_cols],
    on=["PLAYER_ID", "game_id_int"], how="left"
)

roll_cols += fp_roll_cols
print(f"fantasy pts rolling features: {fp_roll_cols}")
print(f"roll_cols total: {len(roll_cols)}")

C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\3233424617.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs[col] = (player_logs.groupby("PLAYER_ID")["FANTASY_PTS"]


C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\3233424617.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs[col] = (player_logs.groupby("PLAYER_ID")["FANTASY_PTS"]


C:\Users\falkt\AppData\Local\Temp\ipykernel_20188\3233424617.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_logs[col] = (player_logs.groupby("PLAYER_ID")["FANTASY_PTS"]


fantasy pts rolling features: ['player_FANTASY_PTS_L3', 'player_FANTASY_PTS_L5', 'player_FANTASY_PTS_L10']
roll_cols total: 75


In [24]:
from sklearn.ensemble import RandomForestRegressor

is_train = df_clean["GAME_DATE"] < "2023-10-01"

Xtr = df_clean.loc[is_train,  roll_cols]
Xte = df_clean.loc[~is_train, roll_cols]
ytr = df_clean.loc[is_train,  "FANTASY_PTS"]
yte = df_clean.loc[~is_train, "FANTASY_PTS"]

# 1. choose model class
# 2. instantiate
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# 3. fit
rf.fit(Xtr, ytr)

# 4. evaluate
tr_rf = np.sqrt(mean_squared_error(ytr, rf.predict(Xtr)))
te_rf = np.sqrt(mean_squared_error(yte, rf.predict(Xte)))
print(f"Random Forest (rolling features) -- train={tr_rf:.3f}  test={te_rf:.3f}")
print(f"Linear baseline                  -- train=9.365  test=9.811")

# top features
importances = pd.Series(rf.feature_importances_, index=roll_cols).sort_values(ascending=False)
print("top 10 features:")
print(importances.head(10).to_string())

Random Forest (rolling features) -- train=3.541  test=9.969
Linear baseline                  -- train=9.365  test=9.811
top 10 features:
player_FANTASY_PTS_L10    0.482137
player_FANTASY_PTS_L3     0.015378
player_FANTASY_PTS_L5     0.012058
player_MIN_L3             0.010001
player_MIN_L10            0.009961
team_reb_L10              0.009198
opp_reb_L10               0.009187
opp_tov_L10               0.008981
team_tov_L10              0.008705
opp_stl_L10               0.008426


## Gradient Boosting on Full Feature Set

All-in: rolling stats + game context + trends + efficiency + missing-teammates + schedule density + Vegas + position + advanced box + DvP.

Switching to `HistGradientBoostingRegressor` because (1) it natively handles NaN â€” critical since Vegas data only covers 2007-2023 and the test set has missing values, and (2) it's much faster than `GradientBoostingRegressor` on this many rows.

In [25]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Assemble feature set. Some bucket lists are empty if the underlying data isn't present yet.
curated_eff_cols = [f"player_{stat}_L{w}" for stat in ["FGA", "FG_PCT", "PLUS_MINUS"] for w in WINDOWS]

gb_cols = (roll_cols
           + context_cols
           + trend_cols
           + curated_eff_cols
           + missing_cols
           + schedule_cols
           + position_cols
           + adv_team_cols
           + dvp_cols)
gb_cols = list(dict.fromkeys(gb_cols))
print(f"total features: {len(gb_cols)}")

# Require non-NaN for core lag-derived features; optional features can be NaN (HistGB handles it)
required = roll_cols + trend_cols + curated_eff_cols
keep = df_clean[required].notna().all(axis=1)
df_gb = df_clean.loc[keep].reset_index(drop=True)
is_train_gb = df_gb["GAME_DATE"] < "2023-10-01"

Xtr3 = df_gb.loc[is_train_gb,  gb_cols]
Xte3 = df_gb.loc[~is_train_gb, gb_cols]
ytr3 = df_gb.loc[is_train_gb,  "FANTASY_PTS"]
yte3 = df_gb.loc[~is_train_gb, "FANTASY_PTS"]
print(f"train rows: {len(Xtr3):,}, test rows: {len(Xte3):,}")

# 1. choose model class (HistGradientBoosting - handles NaN, fast on large data)
# 2. instantiate
gb = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
)
# 3. fit
gb.fit(Xtr3, ytr3)

# 4. evaluate
tr_gb = np.sqrt(mean_squared_error(ytr3, gb.predict(Xtr3)))
te_gb = np.sqrt(mean_squared_error(yte3, gb.predict(Xte3)))
print()
print(f"HistGradientBoosting (full feature set) -- train={tr_gb:.3f}  test={te_gb:.3f}")
print(f"Random Forest (rolling only)            -- train={tr_rf:.3f}  test={te_rf:.3f}")
print(f"Linear baseline                         -- train=9.365  test=9.811")


total features: 107


train rows: 496,750, test rows: 65,882



HistGradientBoosting (full feature set) -- train=9.061  test=9.559
Random Forest (rolling only)            -- train=3.541  test=9.969
Linear baseline                         -- train=9.365  test=9.811


### Feature-Group Ablation

Quantifying how much each feature bucket contributes. Train one HistGB per bucket configuration, starting from the rolling-only baseline and adding one group at a time. The delta in test RMSE shows each bucket's marginal value.

In [26]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Incremental ablation: start with rolling, then add one bucket at a time
configs = [
    ("rolling only",                   roll_cols),
    ("+ context",                      roll_cols + context_cols),
    ("+ trend",                        roll_cols + context_cols + trend_cols),
    ("+ efficiency",                   roll_cols + context_cols + trend_cols + curated_eff_cols),
    ("+ missing teammates",            roll_cols + context_cols + trend_cols + curated_eff_cols + missing_cols),
    ("+ schedule density",             roll_cols + context_cols + trend_cols + curated_eff_cols + missing_cols + schedule_cols),
    ("+ position",                     roll_cols + context_cols + trend_cols + curated_eff_cols + missing_cols + schedule_cols + position_cols),
    ("+ advanced box",                 roll_cols + context_cols + trend_cols + curated_eff_cols + missing_cols + schedule_cols + position_cols + adv_team_cols),
    ("+ DvP (full)",                   roll_cols + context_cols + trend_cols + curated_eff_cols + missing_cols + schedule_cols + position_cols + adv_team_cols + dvp_cols),
]

results = []
prev_test = None
for name, cols in configs:
    cols = list(dict.fromkeys(cols))
    if not cols:
        continue
    Xtr_a = df_gb.loc[is_train_gb,  cols]
    Xte_a = df_gb.loc[~is_train_gb, cols]

    m = HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.05, max_depth=8,
        min_samples_leaf=20, random_state=42,
    )
    m.fit(Xtr_a, ytr3)
    tr = np.sqrt(mean_squared_error(ytr3, m.predict(Xtr_a)))
    te = np.sqrt(mean_squared_error(yte3, m.predict(Xte_a)))
    delta = "" if prev_test is None else f"  Δ {te - prev_test:+.3f}"
    print(f"{name:25s} {len(cols):3d} feats  train={tr:.3f}  test={te:.3f}{delta}")
    results.append({"config": name, "n_features": len(cols), "train_rmse": tr, "test_rmse": te})
    prev_test = te

ablation_df = pd.DataFrame(results)

rolling only               75 feats  train=9.300  test=9.870


+ context                  78 feats  train=9.313  test=9.869  Δ -0.000


+ trend                    86 feats  train=9.296  test=9.865  Δ -0.005


+ efficiency               95 feats  train=9.283  test=9.847  Δ -0.018


+ missing teammates        98 feats  train=9.095  test=9.585  Δ -0.262


+ schedule density        101 feats  train=9.094  test=9.586  Δ +0.001


+ position                106 feats  train=9.086  test=9.576  Δ -0.010


+ advanced box            106 feats  train=9.086  test=9.576  Δ +0.000


+ DvP (full)              107 feats  train=9.061  test=9.559  Δ -0.017


## Hyperparameter Tuning

`RandomizedSearchCV` over the HistGB hyperparameter space using `TimeSeriesSplit(3)`. The grid covers learning rate, max depth, min samples per leaf, max leaf nodes, and L2 regularization. Set `RUN_TUNING = True` to enable - this can take 30+ minutes on the full dataset, so it's off by default.

In [27]:
RUN_TUNING = False  # already ran - results: best CV 9.198, test 9.538
print("Tuning skipped (results from prior run cached in markdown above)")


Tuning skipped (results from prior run cached in markdown above)


## Multi-Output Decomposition

Instead of predicting `FANTASY_PTS` directly, predict each component stat separately (PTS, REB, AST, STL, BLK, TOV, FG3M) and combine via the DK formula. Each model can specialize on the patterns specific to that stat - rebounds depend on size, assists on role, blocks on position - and the DK weights amplify the most important components.

In [28]:
DK_WEIGHTS = {
    "PTS":  1.00,
    "REB":  1.25,
    "AST":  1.50,
    "STL":  2.00,
    "BLK":  2.00,
    "TOV": -0.50,
    "FG3M": 0.50,
}

# We need each stat as a target column on df_gb. They live in player_logs.
target_stats = list(DK_WEIGHTS.keys())
df_gb_targets = df_gb.merge(
    player_logs[["PLAYER_ID", "game_id_int"] + target_stats],
    on=["PLAYER_ID", "game_id_int"], how="left", suffixes=("", "_y"),
)
# pandas adds suffix only if column exists in both; drop any duplicates
for s in target_stats:
    dup = f"{s}_y"
    if dup in df_gb_targets.columns:
        df_gb_targets[s] = df_gb_targets[dup]
        df_gb_targets = df_gb_targets.drop(columns=[dup])

# Train one HistGB per stat
predictions_train = np.zeros_like(ytr3, dtype=float)
predictions_test  = np.zeros_like(yte3, dtype=float)
per_stat_results = []

for stat, weight in DK_WEIGHTS.items():
    ytr_s = df_gb_targets.loc[is_train_gb, stat].astype(float)
    yte_s = df_gb_targets.loc[~is_train_gb, stat].astype(float)

    m = HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.05, max_depth=8,
        min_samples_leaf=20, random_state=42,
    )
    m.fit(Xtr3, ytr_s)
    pred_tr = m.predict(Xtr3)
    pred_te = m.predict(Xte3)
    rmse_tr = np.sqrt(mean_squared_error(ytr_s, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(yte_s, pred_te))
    per_stat_results.append({"stat": stat, "weight": weight, "train_rmse": rmse_tr, "test_rmse": rmse_te})

    predictions_train += weight * pred_tr
    predictions_test  += weight * pred_te

    print(f"  {stat:5s} (w={weight:+.2f})  train RMSE={rmse_tr:.3f}  test RMSE={rmse_te:.3f}")

# Combined fantasy points prediction
fp_train_rmse = np.sqrt(mean_squared_error(ytr3, predictions_train))
fp_test_rmse  = np.sqrt(mean_squared_error(yte3, predictions_test))

print(f"\n=== Fantasy points from decomposition ===")
print(f"Multi-output decomposition  -- train={fp_train_rmse:.3f}  test={fp_test_rmse:.3f}")
print(f"Direct HistGB prediction    -- train={tr_gb:.3f}  test={te_gb:.3f}")

  PTS   (w=+1.00)  train RMSE=5.681  test RMSE=6.104


  REB   (w=+1.25)  train RMSE=2.560  test RMSE=2.541


  AST   (w=+1.50)  train RMSE=1.757  test RMSE=1.913


  STL   (w=+2.00)  train RMSE=0.955  test RMSE=0.970


  BLK   (w=+2.00)  train RMSE=0.786  test RMSE=0.784


  TOV   (w=-0.50)  train RMSE=1.225  test RMSE=1.214


  FG3M  (w=+0.50)  train RMSE=1.016  test RMSE=1.307

=== Fantasy points from decomposition ===
Multi-output decomposition  -- train=9.054  test=9.554
Direct HistGB prediction    -- train=9.061  test=9.559


## Final Results

Putting all the modeling iterations side-by-side.

In [29]:
summary = pd.DataFrame([
    {"model": "Linear Regression baseline (240 raw lag feats)",  "test_rmse": 9.811},
    {"model": "Random Forest (75 rolling feats)",                "test_rmse": te_rf},
    {"model": "HistGB (full feature set, default params)",        "test_rmse": te_gb},
    {"model": "Multi-output decomposition (PTS/REB/AST/STL/BLK/TOV/FG3M)", "test_rmse": fp_test_rmse},
])
summary["delta_vs_linear"] = (summary["test_rmse"] - 9.811).round(3)
summary["pct_improvement"] = ((9.811 - summary["test_rmse"]) / 9.811 * 100).round(2)
summary = summary.sort_values("test_rmse").reset_index(drop=True)
print(summary.to_string(index=False))

                                                    model  test_rmse  delta_vs_linear  pct_improvement
Multi-output decomposition (PTS/REB/AST/STL/BLK/TOV/FG3M)   9.553560           -0.257             2.62
                HistGB (full feature set, default params)   9.558623           -0.252             2.57
           Linear Regression baseline (240 raw lag feats)   9.811000            0.000             0.00
                         Random Forest (75 rolling feats)   9.968768            0.158            -1.61
